# Task 4: LSTM
LSTM is a neural network that reads the message word by word
and remembers what matters.
We use Google Colab because the libraries need a GPU.

In [1]:
import tensorflow as tf
import pandas as pd
import numpy as np
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
train = pd.read_csv('train_prep_v2.csv')
test = pd.read_csv('test_prep_v2.csv')
print(train.shape, test.shape)
print(train['label'].value_counts().to_dict())

TensorFlow: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
(4102, 9) (1026, 9)
{'ham': 3601, 'spam': 501}


## 1. Turn the words into numbers
A neural network cannot read words, it needs numbers.
So we give every word a number: the most common word gets 1,
the next gets 2, and so on.
We learn these numbers from the train data only.
A word in the test data that we never saw gets a special number for "unknown".

In [2]:
from tensorflow.keras.preprocessing.text import Tokenizer
tok = Tokenizer(oov_token='<UNK>', filters='')
tok.fit_on_texts(train['clean_text'])
print('Different words:', len(tok.word_index))
print('First 10 words:', list(tok.word_index.items())[:10])

Different words: 7304
First 10 words: [('<UNK>', 1), ('i', 2), ('to', 3), ('you', 4), ('a', 5), ('the', 6), ('u', 7), ('and', 8), ('in', 9), ('me', 10)]


## 2. How long is a message?
The LSTM needs all the messages to have the same length.
So we must choose a length.
If it is too short, we cut the end of long messages and lose words.
If it is too long, we add many empty places for nothing.
We look at the train data to choose.

In [3]:
lengths = train['clean_text'].str.split().str.len()
print(lengths.describe().round(1))
for n in [30, 40, 50, 60, 80]:
    print(n, 'words covers', round(100 * (lengths <= n).mean(), 1), '% of messages')

count    4102.0
mean       15.6
std        11.1
min         1.0
25%         7.0
50%        12.0
75%        22.0
max       189.0
Name: clean_text, dtype: float64
30 words covers 92.4 % of messages
40 words covers 98.2 % of messages
50 words covers 98.9 % of messages
60 words covers 99.3 % of messages
80 words covers 99.8 % of messages


## 3. Make all messages the same length
We choose 40 words, because it covers 98.2% of the messages.
- A short message gets zeros at the end, until it has 40 numbers.
- A long message is cut after 40 words.
We also turn the labels into numbers: spam = 1, ham = 0.

In [4]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
MAX_LEN = 40
X_train = pad_sequences(tok.texts_to_sequences(train['clean_text']),
                        maxlen=MAX_LEN, padding='post', truncating='post')
X_test = pad_sequences(tok.texts_to_sequences(test['clean_text']),
                       maxlen=MAX_LEN, padding='post', truncating='post')
y_train = (train['label'] == 'spam').astype(int).values
y_test = (test['label'] == 'spam').astype(int).values
print(X_train.shape, X_test.shape)
print('First message:', train['clean_text'][0])
print('As numbers   :', X_train[0])

(4102, 40) (1026, 40)
First message: hey we can go jazz power yoga hip hop kb and yogasana
As numbers   : [ 128   38   30   49 1742 1109 1110 2230 1257 1111    8 1453    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0]


## 4. Keep a part of the train data for checking
We cut a small part from the train data: the validation part.
While the model learns, we check it on this part after every round.
This tells us if the model is learning or just memorizing.
We do not touch the test data until the very end.
We keep the same share of spam in both parts.

In [5]:
from sklearn.model_selection import train_test_split
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)
print('Fit  :', X_fit.shape, '| spam:', y_fit.sum())
print('Valid:', X_val.shape, '| spam:', y_val.sum())

Fit  : (3281, 40) | spam: 401
Valid: (821, 40) | spam: 100


## 5. Build the LSTM
Our model has four layers:
1. Embedding: gives every word a small vector of numbers.
   Words used in the same way get close vectors. The model learns these vectors itself.
2. LSTM: reads the words one by one, from the first to the last.
   It keeps a memory of what it read, and it decides what to keep and what to forget.
   At the end it gives one vector for the whole message.
3. Dropout: during training, it turns off some numbers at random,
   so the model does not memorize the train data.
4. Dense with sigmoid: gives one number between 0 and 1.
   Near 1 means spam, near 0 means ham.

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense, Input
tf.random.set_seed(42)
np.random.seed(42)
vocab_size = len(tok.word_index) + 1
model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(vocab_size, 32),
    LSTM(32),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])
model.compile(loss='binary_crossentropy', optimizer='adam',
              metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 40, 32)         │       233,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 242,113 (945.75 KB)

 Trainable params: 242,113 (945.75 KB)

 Non-trainable params: 0 (0.00 B)

## 6. Train the model
The model reads the train part many times. Each time is called an epoch.
After every epoch we check it on the validation part.
We use EarlyStopping: if the validation loss stops getting better
for 3 epochs, we stop and go back to the best epoch.
This saves time and stops the model from memorizing the data.

In [8]:
from tensorflow.keras.callbacks import EarlyStopping
stop = EarlyStopping(monitor='val_loss', patience=3,
                     restore_best_weights=True)
history = model.fit(X_fit, y_fit,
                    validation_data=(X_val, y_val),
                    epochs=20, batch_size=32,
                    callbacks=[stop], verbose=1)

Epoch 1/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9006 - loss: 0.3221 - val_accuracy: 0.9671 - val_loss: 0.1440
Epoch 2/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9732 - loss: 0.1071 - val_accuracy: 0.9464 - val_loss: 0.1375
Epoch 3/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9890 - loss: 0.0530 - val_accuracy: 0.9829 - val_loss: 0.0771
Epoch 4/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9966 - loss: 0.0208 - val_accuracy: 0.9817 - val_loss: 0.0885
Epoch 5/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9973 - loss: 0.0176 - val_accuracy: 0.9683 - val_loss: 0.1521
Epoch 6/20
103/103 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9936 - loss: 0.0311 - val_accuracy: 0.9805 - val_loss: 0.0914


### Check that we kept the best epoch
EarlyStopping should give us back the weights of the best epoch,
not the last one. We check this: the loss on the validation part now
must be the same as the loss of the best epoch.

In [9]:
h = history.history
print('Number of epochs:', len(h['loss']))
best = int(np.argmin(h['val_loss'])) + 1
print('Best epoch (lowest val_loss):', best)
print('Its val_loss:', round(h['val_loss'][best - 1], 4))
print('Its val_accuracy:', round(h['val_accuracy'][best - 1], 4))
print('Loss now on validation:', round(model.evaluate(X_val, y_val, verbose=0)[0], 4))

Number of epochs: 6
Best epoch (lowest val_loss): 3
Its val_loss: 0.0771
Its val_accuracy: 0.9829
Loss now on validation: 0.0771


## 7. Test the model
Now we use the test data for the first time.
The model gives a number between 0 and 1 for every message.
If the number is 0.5 or more, we say spam.

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix, classification_report
prob = model.predict(X_test, verbose=0).flatten()
pred = (prob >= 0.5).astype(int)
print('Accuracy :', round(100 * accuracy_score(y_test, pred), 1))
print('Precision:', round(100 * precision_score(y_test, pred), 1))
print('Recall   :', round(100 * recall_score(y_test, pred), 1))
print('F1       :', round(100 * f1_score(y_test, pred), 1))
cm = confusion_matrix(y_test, pred)
print(pd.DataFrame(cm, index=['true ham', 'true spam'],
                   columns=['said ham', 'said spam']))

Accuracy : 98.9
Precision: 95.2
Recall   : 96.0
F1       : 95.6
           said ham  said spam
true ham        895          6
true spam         5        120


### Save the predictions
We save what the model said about every test message,
with the same columns as pred_nb.csv and pred_lr.csv.

In [11]:
out = test[['label', 'text']].copy()
out['pred'] = np.where(pred == 1, 'spam', 'ham')
out['prob_spam'] = prob.round(4)
out.to_csv('pred_lstm.csv', index=False)
print(out['pred'].value_counts().to_dict())
out.head()

{'ham': 900, 'spam': 126}


,label,text,pred,prob_spam
0,ham,"sir, you will receive the account no another 1...",ham,0.0061
1,ham,You at mu? You should try to figure out how mu...,ham,0.0240
2,ham,The greatest test of courage on earth is to be...,ham,0.0062
3,ham,Gud mrng dear hav a nice day,ham,0.0047
4,ham,I wonder if you'll get this text?,ham,0.0048


## 8. Save the predictions and the training history
Abdulrahman needs the predictions to compare the models.
Abdullah needs the history to draw the learning curves.

In [12]:
hist = pd.DataFrame(history.history)
hist.index = range(1, len(hist) + 1)
hist.index.name = 'epoch'
hist.round(4).to_csv('lstm_history.csv')
print(hist.round(4))

       accuracy    loss  val_accuracy  val_loss
epoch                                          
1        0.9006  0.3221        0.9671    0.1440
2        0.9732  0.1071        0.9464    0.1375
3        0.9890  0.0530        0.9829    0.0771
4        0.9966  0.0208        0.9817    0.0885
5        0.9973  0.0176        0.9683    0.1521
6        0.9936  0.0311        0.9805    0.0914
